In [6]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import timm


class SkinDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)

        
        self.data = self.data.dropna()
        self.data = self.data[self.data["Disease"] != ""]

        self.transform = transform

        
        self.classes = sorted(self.data["Disease"].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]["Image_Path"]
        label_name = self.data.iloc[idx]["Disease"]
        label = self.class_to_idx[label_name]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label



transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])



dataset = SkinDataset("your_file.csv", transform=transform)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

print("Classes:", dataset.classes)



model = timm.create_model('vit_base_patch16_224', pretrained=True)


model.head = torch.nn.Linear(model.head.in_features, len(dataset.classes))



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)



epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")



torch.save(model.state_dict(), "vit_skin_model.pth")

ModuleNotFoundError: No module named 'torch'